<a href="https://colab.research.google.com/github/s-rafia/voice-controlled-design-tool/blob/main/notebooks/export_to_onnx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

MODEL_DIR = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/model'
ONNX_DIR  = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/onnx_model'

model = ORTModelForSequenceClassification.from_pretrained(MODEL_DIR, export=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model.save_pretrained(ONNX_DIR)
tokenizer.save_pretrained(ONNX_DIR)

import os
print(os.listdir(ONNX_DIR))

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


['config.json', 'model.onnx', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']


In [3]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

source = ONNX_DIR + '/model.onnx'
target = ONNX_DIR + '/model_quantized.onnx'

print('before:', round(os.path.getsize(source) / 1e6, 1), 'MB')

quantize_dynamic(source, target, weight_type=QuantType.QUInt8)

print('after: ', round(os.path.getsize(target) / 1e6, 1), 'MB')

before: 268.0 MB


after:  67.4 MB


In [4]:
import numpy as np
import pandas as pd
import json
import onnxruntime as ort
from transformers import AutoTokenizer

DATA_DIR = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/Data'

tokenizer = AutoTokenizer.from_pretrained(ONNX_DIR)

maps = json.load(open(MODEL_DIR + '/label_maps.json'))
labels = maps['labels']

hard = pd.read_csv(DATA_DIR + '/hard_eval.csv')


def accuracy_of(onnx_path):
    session = ort.InferenceSession(onnx_path)

    input_names = []
    for spec in session.get_inputs():
        input_names.append(spec.name)

    correct = 0
    for i in range(len(hard)):
        phrase = hard['phrase'][i]
        true_label = hard['label'][i]

        encoded = tokenizer(phrase, return_tensors='np', truncation=True,
                            padding='max_length', max_length=32)

        inputs = {}
        for name in input_names:
            inputs[name] = encoded[name].astype(np.int64)

        logits = session.run(None, inputs)[0]
        predicted = int(np.argmax(logits, axis=1)[0])

        if labels[predicted] == true_label:
            correct = correct + 1

    return correct / len(hard)


full = accuracy_of(ONNX_DIR + '/model.onnx')
small = accuracy_of(ONNX_DIR + '/model_quantized.onnx')

print('full size  :', round(full * 100, 1), '%')
print('quantised  :', round(small * 100, 1), '%')

full size  : 93.3 %
quantised  : 95.0 %


In [5]:
import shutil
import os

WEB_MODEL_DIR = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/web_model'
os.makedirs(WEB_MODEL_DIR + '/onnx', exist_ok=True)

for name in ['config.json', 'tokenizer.json', 'tokenizer_config.json',
             'special_tokens_map.json', 'vocab.txt']:
    shutil.copy(ONNX_DIR + '/' + name, WEB_MODEL_DIR + '/' + name)

shutil.copy(ONNX_DIR + '/model_quantized.onnx',
            WEB_MODEL_DIR + '/onnx/model_quantized.onnx')

print(os.listdir(WEB_MODEL_DIR))
print(os.listdir(WEB_MODEL_DIR + '/onnx'))

shutil.make_archive('/content/web_model', 'zip', WEB_MODEL_DIR)
print('zip:', round(os.path.getsize('/content/web_model.zip') / 1e6, 1), 'MB')

['onnx', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt']
['model_quantized.onnx']
zip: 43.6 MB
